In [ ]:
import pandas as pd
import numpy as np
import os   
import time
import h5py
import warnings
import logging

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from itertools import product
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR




def split_datasets_random(datasets, train_ratio=0.8, seed=42):
    train_sets = {}
    test_sets = {}
    
    # This loop keeps dataset structure intact
    for name, df in datasets.items():
        if "Dataset" not in df.columns:
            raise ValueError(f"Dataset '{name}' is missing 'Dataset' column")
        
        # Get all unique dataset names in this dataframe
        all_names = df["Dataset"].unique().tolist()
        
        # Shuffle identically to split_h5()
        np.random.seed(seed)
        np.random.shuffle(all_names)
        
        # Split dataset names into train/test sets
        n_train = int(len(all_names) * train_ratio)
        train_names = set(all_names[:n_train])
        test_names = set(all_names[n_train:])
        
        # Assign rows based on dataset name
        train_df = df[df["Dataset"].isin(train_names)].reset_index(drop=True)
        test_df = df[df["Dataset"].isin(test_names)].reset_index(drop=True)
        
        train_sets[name] = train_df
        test_sets[name] = test_df
    
    return train_sets, test_sets

def prepare_for_autogluon(df):
    """
    Prepare dataframe for AutoGluon by dropping only the 'Dataset' column.

    Parameters:
        df (pd.DataFrame): input dataframe

    Returns:
        pd.DataFrame ready for AutoGluon (features + target only)
    """
    return df.drop(columns=["Dataset"], errors='ignore')

def autogluon_automation(train_df, test_df, dataset_name, target, 
                         base_path="Autogluon Models",
                         presets="medium_quality", 
                         metric="r2",
                         num_bag_folds=5, 
                         num_stack_levels=1, 
                         verbosity=1):
    
    start_time = time.time()  # Start timer
    
    # Drop other target columns if they exist
    other_targets = [col for col in ["CO_tbu", "MP_CF3"] if col != target]
    train_df = train_df.drop(columns=other_targets, errors="ignore")
    test_df = test_df.drop(columns=other_targets, errors="ignore")

    assert target in train_df.columns, f"{target} missing from train_df!"
    assert target in test_df.columns, f"{target} missing from test_df!"
    for col in other_targets:
        assert col not in train_df.columns, f"{col} still in train_df!"
        assert col not in test_df.columns, f"{col} still in test_df!"

    # Create folder hierarchy with target as top-level
    model_name = f"{target}_{dataset_name}_{presets}"
    model_path = os.path.join(base_path, target, dataset_name, presets, model_name)

    # Load the model if it already exists (so results are still reported); otherwise train it
    model_loaded = os.path.exists(model_path) and os.listdir(model_path)
    if model_loaded:
        print(f"Model at '{model_path}' exists. Loading it instead of retraining.")
        predictor = TabularPredictor.load(model_path)
    else:
        os.makedirs(model_path, exist_ok=True)

        predictor = TabularPredictor(label=target, path=model_path, eval_metric=metric).fit(
            train_data=train_df,
            presets=presets,
            num_bag_folds=num_bag_folds,
            num_stack_levels=num_stack_levels,
            verbosity=verbosity
        )

    # Get CV R² explicitly regardless of optimization metric
    lb = predictor.leaderboard(silent=True)

    if 'score_val' in lb.columns:
        # Sort leaderboard by score_val (descending)
        lb_sorted = lb.sort_values(by='score_val', ascending=False)
        cv_r2 = lb_sorted.iloc[0]['score_val']
    else:
        cv_r2 = None

    # Evaluate on test set
    test_results = predictor.evaluate(test_df, silent=True)

    # Extract R2 and RMSE helper
    def extract_metrics(results):
        r2 = results.get('r2', None)
        rmse = results.get('root_mean_squared_error', None)
        if rmse is None:
            rmse = results.get('rmse', None)
        return r2, rmse

    test_r2, test_rmse = extract_metrics(test_results)

    elapsed_time = None if model_loaded else time.time() - start_time  # Training time (None for loaded models)

    # Flip sign of RMSE
    test_rmse_flipped = -test_rmse if test_rmse is not None else None

    results = {
        'Dataset': dataset_name, 'Preset': presets, 'Metric': metric, 'Target': target,
        'CV_R2': cv_r2,
        'Test_R2': test_r2, 'Test_RMSE': test_rmse_flipped,
        'Elapsed_Time_sec': elapsed_time
    }

    print(f"Results for {dataset_name} | Target: {target} | Preset: {presets} | Metric: {metric}:")
    print(f"  CV R2:    {cv_r2:.3f}" if cv_r2 is not None else "  CV R2:    N/A")
    print(f"  Test R2:  {test_r2:.3f}, Test RMSE:  {test_rmse_flipped:.3f}")
    print(f"  Elapsed Time: {elapsed_time:.2f} seconds\n" if elapsed_time is not None else "  Elapsed Time: N/A (loaded existing model)\n")

    return results

def batch_train_autogluon(train_sets, test_sets, targets, presets_list,
                          base_path="AutogluonModels",
                          num_bag_folds=5, num_stack_levels=1, verbosity=2):
    results_list = []

    for dataset_name in train_sets.keys():
        train_df = train_sets[dataset_name]
        test_df = test_sets[dataset_name]

        for target in targets:
            for preset in presets_list:

                print(f"\n=== Training: Dataset={dataset_name}, Target={target}, Preset={preset} ===")
                
                res = autogluon_automation(
                    train_df=train_df,
                    test_df=test_df,
                    dataset_name=dataset_name,
                    target=target,
                    base_path=base_path,
                    presets=preset,
                    metric='r2',
                    num_bag_folds=num_bag_folds,
                    num_stack_levels=num_stack_levels,
                    verbosity=verbosity
                )
                
                # Append result only if training happened (res is not None)
                if res is not None:
                    results_list.append(res)

    # Convert results list of dicts into a DataFrame
    return pd.DataFrame(results_list)


In [ ]:
datasets = {
    "Dataset 1": pd.read_csv("Featurized datasets/Training Datasets/Dataset 1.csv"),
    "Dataset 2": pd.read_csv("Featurized datasets/Training Datasets/Dataset 2.csv"),
    "Dataset 3": pd.read_csv("Featurized datasets/Training Datasets/Dataset 3.csv"),
    "Dataset 4": pd.read_csv("Featurized datasets/Training Datasets/Dataset 4.csv"),
    "Dataset 5": pd.read_csv("Featurized datasets/Training Datasets/Dataset 5.csv"),
}

# Random Split Datasets

In [ ]:
random_split_train, random_split_test = split_datasets_random(datasets)
random_split_train = {name: prepare_for_autogluon(df) for name, df in random_split_train.items()}
random_split_test = {name: prepare_for_autogluon(df) for name, df in random_split_test.items()}

In [ ]:
targets = ["CO_tbu","MP_CF3"]
presets_list = ["medium_quality","best_quality"]

results_df = batch_train_autogluon(random_split_train, random_split_test, targets, presets_list, 
                                   verbosity = 0, base_path= "Autogluon Models")

print(results_df)
results_df.to_excel("Autogluon Results/Random Split Autogluon Results.xlsx", index=False)

# One Hot Encoding

In [ ]:
one_hot_dataset = {"One Hot Encoding": pd.read_csv("Featurized datasets/Training Datasets/One-hot encoded dataset.csv")}

In [ ]:
one_hot_train, one_hot_test = split_datasets_random(one_hot_dataset)
one_hot_train = {name: prepare_for_autogluon(df) for name, df in one_hot_train.items()}
one_hot_test = {name: prepare_for_autogluon(df) for name, df in one_hot_test.items()}

In [ ]:
targets = ["CO_tbu","MP_CF3"]
presets_list = ["medium_quality","best_quality"]
OHE_results = batch_train_autogluon(one_hot_train, one_hot_test, targets, presets_list,
                     verbosity=0, base_path="Autogluon Models/One Hot Encoding")

OHE_results.to_excel("Autogluon Results/One Hot Encoding Results.xlsx", index=False)

# Benchmarking Autogluon vs other algorithms

In [ ]:
autogluon_model_CO_ds4 = TabularPredictor.load("Autogluon Models/CO_tbu/Dataset 4/best_quality/CO_tbu_Dataset 4_best_quality")
autogluon_model_MP_ds4 = TabularPredictor.load("Autogluon Models/MP_CF3/Dataset 4/best_quality/MP_CF3_Dataset 4_best_quality")

In [ ]:
train_CO_ds4 = random_split_train["Dataset 4"].drop(columns=["MP_CF3"], errors="ignore")
test_CO_ds4 = random_split_test["Dataset 4"].drop(columns=["MP_CF3"], errors="ignore")

train_MP_ds4 = random_split_train["Dataset 4"].drop(columns=["CO_tbu"], errors="ignore")
test_MP_ds4 = random_split_test["Dataset 4"].drop(columns=["CO_tbu"], errors="ignore")

print(train_CO_ds4.shape, test_CO_ds4.shape)
print(train_MP_ds4.shape, test_MP_ds4.shape)

In [ ]:
# Suppress all warnings and logs during model evaluation
warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)

def evaluate_all_models_silent(train_df, test_df, label, random_state=42, n_splits=5, n_estimators=200):
    """
    Same as evaluate_all_models but suppresses all intermediate output.
    Only returns results dict for summary table compilation.
    """
    X_train = train_df.drop(columns=[label])
    y_train = train_df[label]
    X_test  = test_df.drop(columns=[label])
    y_test  = test_df[label]

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    model_dict = {
        "Random Forest": RandomForestRegressor(
            n_estimators=n_estimators,
            random_state=random_state,
            oob_score=True
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=n_estimators,
            random_state=random_state
        ),
        "SVR": Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR())
        ]),
        "Linear Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LinearRegression())
        ])
    }

    results = []
    fitted_models = {}

    for name, model in model_dict.items():
        cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2')
        model.fit(X_train, y_train)
        fitted_models[name] = model

        oob_r2 = round(model.oob_score_, 3) if name == "Random Forest" else None

        y_pred_test = model.predict(X_test)
        r2_test   = r2_score(y_test, y_pred_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
        mae_test  = mean_absolute_error(y_test, y_pred_test)

        results.append({
            "Model":        name,
            "Label":        label,
            "CV R² (mean)": round(cv_scores.mean(), 3),
            "CV R² (std)":  round(cv_scores.std(), 3),
            "OOB R²":       oob_r2,
            "Test R²":      round(r2_test, 3),
            "Test RMSE":    round(rmse_test, 3),
            "Test MAE":     round(mae_test, 3)
        })

    return pd.DataFrame(results), fitted_models


# Run evaluations silently
print("Evaluating C-O coupling models...")
summary_CO, models_CO = evaluate_all_models_silent(
    train_df=train_CO_ds4,
    test_df=test_CO_ds4,
    label="CO_tbu"
)

print("Evaluating Metallaphotoredox models...")
summary_MP, models_MP = evaluate_all_models_silent(
    train_df=train_MP_ds4,
    test_df=test_MP_ds4,
    label="MP_CF3"
)

# Compile and print final summary table only
master_summary = pd.concat([summary_CO, summary_MP], ignore_index=True)
print("\n=== MASTER SUMMARY TABLE ===")
print(master_summary.to_string(index=False))

In [ ]:
def predict_and_save_top_sklearn(
    fitted_models,
    target_name,
    library_df,
    label,
    model_names=["Random Forest", "Gradient Boosting"],
    top_n=10,
    output_csv_path=None
):
    results = {}

    # Drop label column and Dataset column if present in library_df
    cols_to_drop = [col for col in ["Dataset", label] if col in library_df.columns]
    features_df = library_df.drop(columns=cols_to_drop)

    for name in model_names:
        if name not in fitted_models:
            print(f"Warning: {name} not found in fitted_models, skipping.")
            continue

        model = fitted_models[name]

        # Generate predictions
        preds = model.predict(features_df)

        # Attach predictions and Dataset column to results
        result_df = library_df[["Dataset"]].copy()
        result_df[target_name] = preds

        # Sort by predicted yield descending
        sorted_df = result_df.sort_values(
            by=target_name, ascending=False
        ).reset_index(drop=True)

        # Select top N
        if top_n is None:
            top_df = sorted_df.copy()
        else:
            top_df = sorted_df.head(top_n).copy()

        top_datasets = top_df["Dataset"].tolist()

        # Save full predictions if path provided
        if output_csv_path:
            csv_path = output_csv_path.replace(".csv", f"_{name.replace(' ', '_')}.csv")
            sorted_df.to_csv(csv_path, index=False)
            print(f"Saved {name} predictions to {csv_path}")

        # Print top predictions
        print(f"\n=== Top {top_n} Predictions: {name} | {target_name} ===")
        print(top_df.to_string(index=False))

        results[name] = {
            "top_datasets": top_datasets,
            "top_df": top_df,
            "full_df": sorted_df
        }

    return results

In [ ]:
# --- Load CSV files into lists ---
full_library_paths = [
    "Featurized datasets/Full Library/Full Library Dataset 1.csv",
    "Featurized datasets/Full Library/Full Library Dataset 2.csv",
    "Featurized datasets/Full Library/Full Library Dataset 3.csv",
    "Featurized datasets/Full Library/Full Library Dataset 4.csv",
    "Featurized datasets/Full Library/Full Library Dataset 5.csv"
]

full_library_OHE_path = "Featurized datasets/Full Library/Full Library one-hot encoded.csv"

full_library_dfs = [pd.read_csv(p) for p in full_library_paths]
full_library_OHE_df = pd.read_csv(full_library_OHE_path)

# Load dataset names from the HDF5 file
training_h5 = "HDF5 Datasets/250708 - Training Dataset.h5"
with h5py.File(training_h5, "r") as f:
    h5_dataset_names = list(f.keys())

# Filtering function
def filter_training_data(df, dataset_col="Dataset"):
    # Keep only rows ending with "_B01"
    df = df[df[dataset_col].str.endswith("_B01", na=False)]
    # Remove datasets already in HDF5 training data
    return df[~df[dataset_col].isin(h5_dataset_names)]


# Apply filtering to all datasets in lists
full_library_dfs = [filter_training_data(df) for df in full_library_dfs]

In [ ]:
# C-O coupling predictions
print("Generating C-O coupling predictions...")
co_sklearn_preds = predict_and_save_top_sklearn(
    fitted_models=models_CO,
    target_name="CO_tbu",
    library_df=full_library_dfs[3],       
    label="CO_tbu",
    model_names=["Random Forest", "Gradient Boosting"],
    top_n=10,
    output_csv_path="Autogluon Predictions/sklearn/CO_tbu_sklearn_predictions.csv"
)

# Metallaphotoredox predictions
print("Generating Metallaphotoredox predictions...")
mp_sklearn_preds = predict_and_save_top_sklearn(
    fitted_models=models_MP,
    target_name="MP_CF3",
    library_df=full_library_dfs[3],      
    label="MP_CF3",
    model_names=["Random Forest", "Gradient Boosting"],
    top_n=10,
    output_csv_path="Autogluon Predictions/sklearn/MP_CF3_sklearn_predictions.csv"
)

# Access results for each model
rf_co_top = co_sklearn_preds["Random Forest"]["top_df"]
gb_co_top = co_sklearn_preds["Gradient Boosting"]["top_df"]

rf_mp_top = mp_sklearn_preds["Random Forest"]["top_df"]
gb_mp_top = mp_sklearn_preds["Gradient Boosting"]["top_df"]

# Random Y-test

In [ ]:
# Only create the Y-randomized dataset once: the Y-test models were trained on the saved file,
# and the shuffle below is unseeded, so re-running it would produce a different dataset
randomized_path = "Featurized datasets/Training Datasets/Dataset 4 randomized.csv"
if os.path.exists(randomized_path):
    print(f"'{randomized_path}' exists. Keeping the saved file.")
else:
    df = pd.read_excel("250728 - GCMS Yields.xlsx")

    # Randomize columns
    df['CO_tbu'] = np.random.permutation(df['CO_tbu'].values)
    df['MP_CF3'] = np.random.permutation(df['MP_CF3'].values)

    # Load Dataset4.csv
    df4 = pd.read_csv("Featurized datasets/Training Datasets/Dataset 4.csv")

    # Replace columns in Dataset4 with the randomized ones
    df4['CO_tbu'] = df['CO_tbu'].values
    df4['MP_CF3'] = df['MP_CF3'].values

    # Save result
    df4.to_csv("Featurized datasets/Training Datasets/Dataset 4 randomized.csv", index=False)


In [ ]:
datasets = {
    "Dataset 4": pd.read_csv("Featurized datasets/Training Datasets/Dataset 4 randomized.csv"),
}
random_split_train, random_split_test = split_datasets_random(datasets)
random_split_train = {name: prepare_for_autogluon(df) for name, df in random_split_train.items()}
random_split_test = {name: prepare_for_autogluon(df) for name, df in random_split_test.items()}

In [ ]:
targets = ["CO_tbu","MP_CF3"]
presets_list = ["best_quality"]
results_df = batch_train_autogluon(random_split_train, random_split_test, targets, presets_list, 
                                   verbosity = 0, base_path= "Autogluon Models Y-test")

print(results_df)
results_df.to_excel("Autogluon Results/Random Split Autogluon Results Y-test.xlsx", index=False)

In [ ]:
targets = ["CO_tbu","MP_CF3"]
presets_list = ["medium_quality"]
results_df = batch_train_autogluon(random_split_train, random_split_test, targets, presets_list, 
                                   verbosity = 0, base_path= "Autogluon Models Y-test")

print(results_df)
results_df.to_excel("Autogluon Results/Random Split Autogluon Results Y-test medium_quality.xlsx", index=False)